In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

# Exploratory Data Analysis


Exploratory Data Analysis on dataset 'SampleSuperstore' to find out the patterns in data and to analyse the business trends in order to determine the weak areas that needs to be worked on in order to make more profit

# Importing Libraries

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd

# Reading the data

In [ ]:
%%time
### cell 0 ###

data=pd.read_csv("/home/dias-benchmarks/notebooks/roopacalistus/retail-supermarket-store-analysis/input/superstore/SampleSuperstore.csv")

# -- STEFANOS -- Replicate Data

In [ ]:
%%time
### cell 1 ###

# factor = 800
factor = 8
data = pd.concat([data]*factor)
data.info()

In [ ]:
%%time
### cell 2 ###

data.head()

In [ ]:
%%time
### cell 3 ###

data.info()

In [ ]:
%%time
### cell 4 ###

data.duplicated().sum()

#### So, there are 17 duplicate entries and let us drop them.

In [7]:
# STEFANOS: Disable this. It undos the data replication.
# data.drop_duplicates(inplace= True)

In [ ]:
%%time
### cell 5 ###

data.info()

# Dropping unwanted columns 

In [ ]:
%%time
### cell 6 ###

data.drop(["Postal Code"], axis=1,inplace= True)

In [ ]:
%%time
### cell 7 ###

data.info()

# Correlation between the data

In [ ]:
%%time
### cell 8 ###

data.corr(numeric_only=True)

In [12]:
# STEFANOS: Disable plotting
# sns.heatmap(data.corr(), annot=True)

# EDA

### Analysing the different kinds of Shipping Modes, Segments and categories mentioned in the data

## Shipping Mode

In [ ]:
%%time
### cell 9 ###

data["Ship Mode"].value_counts()

In [14]:
# STEFANOS: Disable plotting
# sns.countplot(x= data['Ship Mode'])

## Different Segments

In [ ]:
%%time
### cell 10 ###

data["Segment"].value_counts()

In [16]:
# STEFANOS: Disable plotting
# sns.countplot(x= data['Segment'])

## Categories of the items

In [ ]:
%%time
### cell 11 ###

data["Category"].value_counts()

In [18]:
# STEFANOS: Disable plotting
# sns.countplot(x= data['Category'])

#### From the above plot we can conclude that Office Supplies Category has highest number of sales. Now let us see the sub-categories as well.


## Sub-categories of items

In [ ]:
%%time
### cell 12 ###

data["Sub-Category"].value_counts()


In [20]:
# STEFANOS: Disable plotting
# plt.figure(figsize=(15,15))
# plt.pie(data["Sub-Category"].value_counts(), labels= data["Sub-Category"].value_counts().index, autopct ="%2f")
# plt.show()

#### Here, Sub-Category with highest sale is Binder, follwed by Paper and Furnishings as second and third respectively.

In [ ]:
%%time
### cell 13 ###

st_profit=data.groupby(["State"])["Profit"].sum().nlargest(20)

In [ ]:
%%time
### cell 14 ###

st_profit

In [23]:
# STEFANOS: Disable plotting
# plt.figure(figsize=(15,8))
# st_profit.plot.bar()

In [24]:
# STEFANOS: Disable plotting
# sns.lineplot(data=data, x="Discount", y= "Profit")

#### So we can see that when discount increases profit decreases

In [25]:
# STEFANOS: Disable plotting
# data.plot(kind="scatter",x="Sales",y="Profit", c="Discount", colormap="Set1",figsize=(10,10))

In this scatter plot we can clearly see that more sales does not mean more profit. It depends on discount as well. 
When Sales is high and there is low discount, the profit margin is higher.

In [ ]:
%%time
### cell 15 ###

data1= data.groupby("State")[["Sales","Profit"]].sum().sort_values(by="Sales", ascending=False)
# STEFANOS: Disable plotting
# data1[:].plot.bar(color = ["Green","Red"], figsize=(20,12))
# plt.title("Profit-Loss and Sales across States")
# plt.show()

California and New York generate more profit compared to the other states.

# Profit-Loss and Sales across Region

In [ ]:
%%time
### cell 16 ###

data1= data.groupby("Region")[["Sales","Profit"]].sum().sort_values(by="Sales", ascending=False)
# STEFANOS: Disable plotting
# data1[:].plot.bar(color = ["Blue","Red"], figsize=(10,7))
# plt.title("Profit-Loss and Sales across Region")
# plt.show()




# Conclusion:


1. The western region generates highest profit.
2. California, NewYork and Washington generates the most sales compared to the other places. 
3. The central region generates lowest profit. 
4. Texas, Pennsylvenia, Florida, Illinois, Ohio and some other states are generating loss with high sale. So we need to give some attention towards them.

Therefore, we have to work more on California and New York. Increase the sales in these states by reducing sales in states like Texas, Florida, Ohio. By decreasing the discount rates in Central region we can increase the profit. Finally we should increase the sale of Office Supplies category as they contribute more.



